# nb01 — анатомия пампа в системе координат групп (G1–G4)

**Откуда постановка.** У практиков (журнал сделок, скриншоты) пампы делятся на
группы по итоговой величине: G1 10–15%, G2 15–20%, G3 20–25%, G4 >25% — и
группа решает **сторону и способ** торговли (в журнале: G2 → SHORT, G4 → LONG).

**Три вопроса этого notebook:**
1. Каков состав: сколько каких пампов, стабилен ли состав по окнам?
2. Анатомия по группам: правда ли фейдить монстров (G3/G4) — самоубийство,
   а мелочь (G1/G2) — прибыльно? Где пик, сколько бежит?
3. **Различима ли будущая группа в моменте** (на пересечении +5%/+10%) по
   каузальным признакам — скорость добегания, откаты, объём?

⚠️ **Группа считается по итоговому забегу (hindsight)** — это инструмент
АНАЛИЗА. Стратегия имеет право использовать только то, что известно на баре:
tL, retrL, surgL, признаки триггера. Прямо торговать «по группе» нельзя,
пока группа не предсказана каузально (это nb02).

Книга: `_out/pump_levels.parquet` (`_build_pump_levels.py`): путь каждого пампа
1440м вперёд от first-trigger входа; runup/t_peak; для каждого уровня +5..+25%:
момент пересечения, откат и объём до него, fade-доходности из точки пересечения
(шорт с 30% катастроф-стопом, выходы +30/60/240м, net издержек).

In [1]:
import sys; sys.path.insert(0, '.')
from _lab import *

X = pd.read_parquet('_out/pump_levels.parquet')
X['entry'] = pd.to_datetime(X['entry'], utc=True)
def window(t):
    if t < pd.Timestamp('2025-07-01', tz='UTC'): return 'TRAIN'
    if t < pd.Timestamp('2026-02-01', tz='UTC'): return 'VALID'
    return 'TEST'
X['win'] = X.entry.map(window)
def group(r):
    if r < 0.10: return 'G0 <10'
    if r < 0.15: return 'G1 10-15'
    if r < 0.20: return 'G2 15-20'
    if r < 0.25: return 'G3 20-25'
    return 'G4 >25'
X['grp'] = X.runup.map(group)
GO = ['G0 <10','G1 10-15','G2 15-20','G3 20-25','G4 >25']

print('pumps:', len(X), '| symbols:', X.sym.nunique())
comp = X.pivot_table(index='grp', columns='win', values='sym', aggfunc='size').reindex(GO)[['TRAIN','VALID','TEST']]
print()
print('=== состав групп по окнам (доля %, в скобках n) ===')
for g in GO:
    r = comp.loc[g]
    tot = {w: comp[w].sum() for w in comp.columns}
    print(f'{g:>9}: ' + '  '.join(f'{w} {r[w]/tot[w]*100:5.1f}% ({int(r[w])})' for w in comp.columns))

pumps: 49836 | symbols: 573

=== состав групп по окнам (доля %, в скобках n) ===
   G0 <10: TRAIN  57.3% (10856)  VALID  51.4% (8550)  TEST  51.1% (7288)
 G1 10-15: TRAIN  14.3% (2710)  VALID  12.6% (2097)  TEST  12.9% (1839)
 G2 15-20: TRAIN   8.6% (1633)  VALID   8.5% (1420)  TEST   9.6% (1375)
 G3 20-25: TRAIN   5.9% (1117)  VALID   5.9% (974)  TEST   5.7% (818)
   G4 >25: TRAIN  13.8% (2617)  VALID  21.6% (3589)  TEST  20.7% (2953)


## 1. Анатомия: пик и время жизни по группам

t_peak = минута максимума в окне 1440м. Если у G1 пик рано, а у G4 поздно —
уже одно это диктует разные горизонты.

In [2]:
t = X.groupby('grp').agg(n=('runup','size'),
        runup_med=('runup', lambda s: round(s.median()*100,1)),
        tpeak_med=('t_peak','median'), tpeak_q90=('t_peak', lambda s: s.quantile(.9)),
        k_med=('k','median'), clen_med=('clen','median')).reindex(GO)
print(t.to_string())

              n  runup_med  tpeak_med  tpeak_q90  k_med  clen_med
grp                                                              
G0 <10    26694        4.0       62.0     1078.0    3.0       3.0
G1 10-15   6646       12.2      517.5     1320.0    5.0       5.0
G2 15-20   4428       17.2      651.5     1340.0    5.0       6.0
G3 20-25   2909       22.3      731.0     1369.0    5.0       6.0
G4 >25     9159       39.9      878.0     1391.0    7.0       9.0


## 2. Сторона по группам: fade из точки пересечения уровня

Читаем как: «шорт в момент, когда памп пересёк +L%, выход через 60м».
Колонки — уровень входа L, строки — итоговая группа. Значение — mean pnl %.

Это АНАТОМИЯ (группа известна задним числом): таблица говорит, ЧТО БЫ БЫЛО,
если бы группу знали. Она проверяет плейбук практиков (фейдить G1/G2,
не фейдить G3/G4), но торговой стратегией станет только после nb02.

In [3]:
for hz in (30, 60, 240):
    print(f'=== fade из пересечения уровня, exit +{hz}м, mean % ===')
    rows = []
    for g in GO:
        sub = X[X.grp == g]
        r = {'grp': g, 'n': len(sub)}
        for L in ('05','10','15','20','25'):
            p = sub[f'fade{L}_{hz}'].dropna()
            r[f'+{L}%'] = round(p.mean()*100,2) if len(p) > 30 else None
        rows.append(r)
    print(pd.DataFrame(rows).set_index('grp').to_string())
    print()

=== fade из пересечения уровня, exit +30м, mean % ===
              n  +05%  +10%  +15%  +20%  +25%
grp                                          
G0 <10    26694  2.36   NaN   NaN   NaN   NaN
G1 10-15   6646  0.49  2.74   NaN   NaN   NaN
G2 15-20   4428 -0.53  0.85  3.27   NaN   NaN
G3 20-25   2909 -1.13 -0.27  1.47  3.78   NaN
G4 >25     9159 -3.65 -3.22 -2.55 -1.57 -0.23

=== fade из пересечения уровня, exit +60м, mean % ===
              n  +05%  +10%  +15%  +20%  +25%
grp                                          
G0 <10    26694  3.29   NaN   NaN   NaN   NaN
G1 10-15   6646  1.03  3.81   NaN   NaN   NaN
G2 15-20   4428 -0.41  1.59  4.46   NaN   NaN
G3 20-25   2909 -1.49  0.14  2.32  5.21   NaN
G4 >25     9159 -5.32 -4.61 -3.55 -2.15 -0.34

=== fade из пересечения уровня, exit +240м, mean % ===
              n   +05%  +10%  +15%  +20%  +25%
grp                                           
G0 <10    26694   5.70   NaN   NaN   NaN   NaN
G1 10-15   6646   2.56  6.39   NaN   NaN   NaN
G2 

## 3. Различима ли группа в моменте пересечения +5% / +10%?

Каузальные величины в точке пересечения: скорость добегания (tL, минут),
глубина отката по пути (retrL), объёмный всплеск (surgL).
Если медианы монотонно расходятся по группам — группа предсказуема онлайн,
и nb02 (walk-forward классификатор) имеет право на жизнь.

In [4]:
for L in ('05','10'):
    sub = X[X[f't{L}'] > 0]
    t = sub.groupby('grp').agg(n=(f't{L}','size'),
            t_med=(f't{L}','median'),
            retr_med=(f'retr{L}', lambda s: round(s.median()*100,2)),
            surg_med=(f'surg{L}', lambda s: round(s.median(),1))).reindex(GO)
    print(f'=== в момент пересечения +{L}%: медианы по группам ===')
    print(t.to_string()); print()

=== в момент пересечения +05%: медианы по группам ===
              n  t_med  retr_med  surg_med
grp                                       
G0 <10    10616  112.0     -4.84      14.5
G1 10-15   6646   74.0     -4.77      13.9
G2 15-20   4428   58.0     -4.78      13.6
G3 20-25   2909   51.0     -5.03      14.5
G4 >25     9159   35.0     -5.27      17.4

=== в момент пересечения +10%: медианы по группам ===
               n  t_med  retr_med  surg_med
grp                                        
G0 <10       NaN    NaN       NaN       NaN
G1 10-15  6645.0  358.0     -8.65      15.4
G2 15-20  4428.0  248.0     -8.74      15.1
G3 20-25  2909.0  193.0     -8.92      15.8
G4 >25    9159.0  105.0     -9.33      21.5



## Выводы nb01

**1. 🟢 Фауна сменилась: доля монстров G4 (>25%) выросла с 13.8% (TRAIN) до
21.6%/20.7% (VALID/TEST).** Это лучший на сегодня кандидат в объяснение смерти
старого памп-фейда в nb00: не «edge выдохся», а «шортящего стало переезжать
в 1.5 раза чаще». Диагноз меняется с «сигнал мёртв» на «нужен фильтр от монстров».

**2. 🟢 Плейбук практиков воспроизведён.** Фейд у вершины мелких групп:
+3.3…+5.2% за 60м (диагональ таблицы §2). Фейд монстра убыточен в ЛЮБОЙ точке:
−5.3% на +5%, до −10.3% на горизонте 240м. «G2 шорти, G4 не трожь» — подтверждено.

**3. 🟢 Безусловный фейд ≈ 0.** Взвешенная по n средняя фейда на пересечении +5%
≈ −0.3%: прибыль малышей полностью съедается монстрами. **Весь edge — в
различении пород; классификатор = стратегия.**

**4. 🟢 Порода различима в моменте, главный признак — скорость.** До +5%
будущий G4 добегает за 35 минут (медиана) против 112 у G0 — монотонная
лестница по всем группам; объёмный всплеск G4 выше (17.4 vs ~14). Откаты
почти не различают (−4.8…−5.3%).

**Оговорки.** Группы считаны по итоговому забегу (hindsight) — таблицы §1–2
описывают анатомию, не стратегию; торговое право имеют только каузальные
величины §3 (t05/retr05/surg05 и признаки триггера). Fade-колонки с 30%
катастроф-стопом ШОРТА — «−fade» это не точный pnl лонга. t_peak медианы
поздние (517–878м) — пик внутри окна 1440м, интерпретировать аккуратно.

**Дальше — nb02:** walk-forward предсказание породы в момент пересечения +5%
и честный расчёт, сколько диагонального edge выживает при реалистичном
качестве предсказания.